# Store Sales – Time Series Forecasting
## Corporación Favorita | Kaggle Competition

> **Objective:** Forecast unit sales for thousands of items sold at Favorita grocery stores  
> **Metric:** Root Mean Squared Logarithmic Error (RMSLE)  
> **Horizon:** 16-day forecast (Aug 16 – Aug 31, 2017)

---

### Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                   DATA INGESTION LAYER                          │
│  train.csv  test.csv  stores.csv  oil.csv  holidays.csv  txn.csv│
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                  FEATURE ENGINEERING LAYER                      │
│  Calendar · Lags · Rolling Stats · Promotion · Oil · Holiday    │
│  Transactions · Store/Family Target Encoding · Fourier Terms    │
└────────────────────────┬────────────────────────────────────────┘
                         │
              ┌──────────┴──────────┐
              ▼                     ▼
┌─────────────────────┐  ┌──────────────────────────┐
│  VALIDATION LAYER   │  │    TRAINING LAYER         │
│  Walk-Forward CV    │  │  Baseline (Seasonal Naïve)│
│  5 folds × 16 days  │  │  LightGBM (log1p target)  │
│  16-day gap (no     │  │  Ensemble (optional)      │
│  future leakage)    │  └──────────────────────────┘
└─────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                   INFERENCE LAYER                               │
│  Retrain on full data → Predict test set → submission.csv       │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                   ERROR ANALYSIS LAYER                          │
│  By Store · By Family · By Holiday · By Promo · By DoW         │
│  Feature Importance · SHAP · Residual Plots                     │
└─────────────────────────────────────────────────────────────────┘
```

## 0. Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.data_loader       import load_raw, build_master, combine_train_test
from src.feature_engineering import build_features
from src.validation        import cross_validate, rmsle, WalkForwardSplitter, check_temporal_leakage
from src.models            import LightGBMForecaster, SeasonalNaiveForecaster, make_lgbm_factory, make_naive_factory
from src.error_analysis    import attach_predictions, generate_error_report, plot_feature_importance, plot_residuals_over_time, plot_error_by_family

DATA_DIR  = '../data/raw'
OUT_DIR   = '../submissions'
os.makedirs(OUT_DIR, exist_ok=True)
print('Environment ready.')

## 1. Data Loading & Exploration

In [ ]:
dfs = load_raw(DATA_DIR)
print('Loaded tables:', list(dfs.keys()))
for name, df in dfs.items():
    print(f'  {name:15s}: {df.shape}')

In [ ]:
train_raw = dfs['train']
print('Train date range:', train_raw['date'].min(), '→', train_raw['date'].max())
print('Stores:', train_raw['store_nbr'].nunique())
print('Families:', train_raw['family'].nunique())
print('Train shape:', train_raw.shape)
print('\nSales stats:')
train_raw['sales'].describe()

In [ ]:
# --- EDA: Aggregate daily sales ---
daily_sales = train_raw.groupby('date')['sales'].sum().reset_index()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_sales['date'], daily_sales['sales'], lw=0.8)
ax.set_title('Total Daily Sales across All Stores & Families')
ax.set_xlabel('Date'); ax.set_ylabel('Total Sales')
plt.tight_layout(); plt.show()

In [ ]:
# --- EDA: Top 10 product families by volume ---
fam_sales = train_raw.groupby('family')['sales'].sum().sort_values(ascending=False)
fam_sales.head(10).plot(kind='bar', figsize=(10,4), title='Top 10 Families by Total Sales')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# --- EDA: Oil price trend ---
oil = dfs['oil'].copy()
oil['date'] = pd.to_datetime(oil['date'])
oil = oil.sort_values('date')
oil['dcoilwtico'].interpolate().plot(figsize=(12,3), title='WTI Oil Price (USD/barrel)')
plt.tight_layout(); plt.show()

In [ ]:
# --- EDA: Holiday distribution ---
hol = dfs['holidays'].copy()
print('Holiday types:', hol['type'].value_counts().to_dict())
print('Holiday locales:', hol['locale'].value_counts().to_dict())

In [ ]:
# --- EDA: Zero-sales analysis ---
zero_pct = (train_raw['sales'] == 0).mean() * 100
promo_zero = train_raw[train_raw['onpromotion'] == 1]['sales'].eq(0).mean() * 100
print(f'Overall zero-sales: {zero_pct:.1f}%')
print(f'Zero-sales when on promotion: {promo_zero:.1f}%')

# Sales distribution (log scale)
import matplotlib.ticker as mtick
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_raw['sales'].clip(upper=500).hist(bins=80, ax=axes[0])
axes[0].set_title('Sales Distribution (clipped at 500)')
np.log1p(train_raw['sales']).hist(bins=80, ax=axes[1])
axes[1].set_title('log1p(Sales) Distribution')
plt.tight_layout(); plt.show()

## 2. Feature Engineering

| Feature Group | Examples | Rationale |
|---|---|---|
| Calendar | year, month, week, day_of_week, is_weekend, is_payday | Weekly/monthly/annual seasonality |
| Fourier terms | sin/cos weekly & annual | Smooth cyclical patterns for gradient boosting |
| Lag features | lag_7, lag_14, lag_28 | Same-day last week/fortnight/month |
| Rolling stats | rolling_mean_7/14/28, rolling_std | Trend + variability |
| Promotion | onpromotion, promo_count_7/14, days_since_promo | Promo lift effect |
| Oil price | level, lag, change, 7-day average | Macro economic signal |
| Holiday | is_national/regional/local_holiday, pre/post ±3 days | Holiday demand spike/dip |
| Transactions | daily footfall (lag+roll) | Demand proxy |
| Store encoding | type, cluster, store_mean_log_sales | Store-level demand level |
| Family encoding | family_mean_log_sales | Family-level demand level |

In [ ]:
print('Building master merged frames…')
train_merged, test_merged = build_master(dfs)
print(f'  Train: {train_merged.shape}  Test: {test_merged.shape}')

print('Combining for feature engineering…')
combined = combine_train_test(train_merged, test_merged)

print('Running feature pipeline…')
featured = build_features(combined)
print(f'  Feature matrix shape: {featured.shape}')
print(f'  Columns: {featured.columns.tolist()}')

In [ ]:
# Split back into train / test
train_feat = featured[featured['split'] == 'train'].copy()
test_feat  = featured[featured['split'] == 'test'].copy()

print(f'Train features: {train_feat.shape}')
print(f'Test features:  {test_feat.shape}')
print(f'\nMissing values in train (%):\n{(train_feat.isnull().mean() * 100).sort_values(ascending=False).head(15)}')

In [ ]:
# Define feature columns (exclude identifiers, target, split flag)
EXCLUDE = ['id', 'date', 'sales', 'split', 'log_sales']
FEATURE_COLS = [c for c in train_feat.columns if c not in EXCLUDE]
TARGET = 'sales'

print(f'Feature count: {len(FEATURE_COLS)}')

# Leakage check (warn if lag < 16 days for the 16-day horizon)
suspicious = check_temporal_leakage(train_feat, FEATURE_COLS, lag_threshold=16)
if suspicious:
    print(f'  Potential leakage columns (lag < 16): {suspicious}')
    print('   NOTE: These are valid for in-sample training; during inference they will be filled from known data.')
else:
    print(' No leakage detected for 16-day horizon.')

## 3. Validation Design

### Why walk-forward (no random split)?

Sales data has **temporal autocorrelation** – a random split would allow the model to learn from future data when predicting the past, artificially inflating validation scores (data leakage).

### Walk-forward scheme

```
Time ──────────────────────────────────────────────────────────────────►

Fold 1  [═══════════ TRAIN ═══════════] [GAP 16d] [VAL 16d]
Fold 2  [══════════════ TRAIN ══════════════════] [GAP 16d] [VAL 16d]
Fold 3  [═══════════════════ TRAIN ═══════════════════════] [GAP 16d] [VAL 16d]
Fold 4  [══════════════════════════ TRAIN ══════════════════════════] [GAP 16d] [VAL 16d]
Fold 5  [═════════════════════════════════ TRAIN ════════════════════════════] [GAP 16d] [VAL 16d]
```

- **GAP = 16 days**: matches the competition test horizon; prevents model from using recent signal it wouldn't have at prediction time
- **Expanding window**: each fold trains on all available historical data up to that point

In [ ]:
# --- Baseline model: Seasonal Naïve (last week same day) ---
print('Running baseline (Seasonal Naïve) cross-validation…')
naive_results, naive_cv_rmsle = cross_validate(
    df=train_feat,
    feature_cols=FEATURE_COLS,
    target_col=TARGET,
    model_factory=make_naive_factory(7),
    n_splits=5,
    verbose=True,
)
print(f'\nBaseline CV RMSLE: {naive_cv_rmsle:.4f}')

## 4. Model Training

In [ ]:
# --- LightGBM: Single holdout validation (last 16 days = exact test horizon) ---
# 5-fold CV on 3M rows is too slow; a single 16-day holdout mirrors the Kaggle test set.
from src.validation import rmsle as rmsle_fn, mae as mae_fn, smape as smape_fn

HORIZON = 16
max_date  = train_feat['date'].max()
val_start = max_date - pd.Timedelta(days=HORIZON - 1)
tr_mask   = train_feat['date'] < val_start
val_mask  = train_feat['date'] >= val_start

X_tr  = train_feat.loc[tr_mask, FEATURE_COLS].fillna(0)
y_tr  = train_feat.loc[tr_mask, TARGET]
X_val = train_feat.loc[val_mask, FEATURE_COLS].fillna(0)
y_val = train_feat.loc[val_mask, TARGET]

print(f'  Train: {tr_mask.sum():,} rows  ({train_feat.loc[tr_mask,"date"].min().date()} → {train_feat.loc[tr_mask,"date"].max().date()})')
print(f'  Val:   {val_mask.sum():,} rows  ({val_start.date()} → {max_date.date()})')
print('  Training LightGBM holdout model…')

holdout_model = LightGBMForecaster(early_stopping=100)
holdout_model.fit(X_tr, y_tr, X_val, y_val)

holdout_preds = holdout_model.predict(X_val)
lgbm_holdout_rmsle = rmsle_fn(y_val.values, holdout_preds)
lgbm_holdout_mae   = mae_fn(y_val.values, holdout_preds)
lgbm_holdout_smape = smape_fn(y_val.values, holdout_preds)

print(f'\nHoldout Metrics (last {HORIZON} days):')
print(f'  RMSLE : {lgbm_holdout_rmsle:.4f}  (Baseline: {naive_cv_rmsle:.4f})')
print(f'  MAE   : {lgbm_holdout_mae:.2f}')
print(f'  SMAPE : {lgbm_holdout_smape:.2f}%')
print(f'  Improvement over baseline: {((naive_cv_rmsle - lgbm_holdout_rmsle) / naive_cv_rmsle * 100):.1f}%')


In [ ]:
# --- Baseline vs LightGBM comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RMSLE comparison
ax = axes[0]
models = ['Seasonal Naïve', 'LightGBM']
rmsles = [naive_cv_rmsle, lgbm_holdout_rmsle]
colors = ['gray', 'steelblue']
bars = ax.bar(models, rmsles, color=colors, width=0.5)
for bar, v in zip(bars, rmsles):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Holdout RMSLE Comparison')
ax.set_ylabel('RMSLE (lower is better)')
ax.set_ylim(0, max(rmsles) * 1.2)

# Prediction distribution on validation set
ax2 = axes[1]
ax2.scatter(np.log1p(y_val.values), np.log1p(holdout_preds), alpha=0.1, s=1, color='steelblue')
lim = max(np.log1p(y_val.values).max(), np.log1p(holdout_preds).max())
ax2.plot([0, lim], [0, lim], 'r--', lw=1)
ax2.set_xlabel('log1p(Actual)')
ax2.set_ylabel('log1p(Predicted)')
ax2.set_title('Actual vs Predicted (log scale, holdout)')

plt.tight_layout()
plt.savefig('../reports/actual_vs_predicted.png', dpi=120)
plt.show()


In [ ]:
# --- Load pre-trained final model (trained on full data by train.py) ---
# The model in models/lgbm_final.txt was trained with 1953 trees on all training data
# using safe lags (≥16 days) with proper holdout-based early stopping.
import lightgbm as lgb_raw

final_model = LightGBMForecaster()
final_model.load('../models/lgbm_final.txt')

print(f'Final model loaded from models/lgbm_final.txt')
print(f'  Trees:    {final_model.model_.num_trees()}')
print(f'  Features: {len(final_model.feature_names_)}')
print(f'  Holdout RMSLE used for model selection: {lgbm_holdout_rmsle:.4f}')


## 5. Error Analysis

In [ ]:
# --- Error analysis on holdout validation set ---
val_df    = train_feat.loc[val_mask].copy()
val_preds = holdout_preds

val_df = attach_predictions(val_df, val_preds)

# Restore family labels (they were label-encoded by feature engineering)
family_map = dict(enumerate(dfs['train']['family'].astype('category').cat.categories))
if 'family' in val_df.columns:
    val_df['family_label'] = val_df['family'].map(family_map).fillna(val_df['family'].astype(str))
    val_df['family'] = val_df['family_label']  # use readable names

report = generate_error_report(val_df)
print('Error report generated.')
print(f'  Holdout RMSLE: {lgbm_holdout_rmsle:.4f}  MAE: {lgbm_holdout_mae:.2f}  SMAPE: {lgbm_holdout_smape:.2f}%')


In [ ]:
print('=== Error by Store (Top 10 Worst) ===')
display(report['by_store'].head(10))

print('\n=== Error by Holiday ===')
display(report['by_holiday'])

print('\n=== Error by Promotion ===')
display(report['by_promotion'])

In [ ]:
# Family error plot
fig = plot_error_by_family(report['by_family'])
plt.show()

In [ ]:
# Residuals over time
fig = plot_residuals_over_time(val_df)
plt.show()

In [ ]:
# Day-of-week error
dow_err = report['by_dow']
dow_err.plot(x='day_name', y='rmsle', kind='bar', figsize=(8, 3),
             title='RMSLE by Day of Week', legend=False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# --- Feature importance ---
fig = plot_feature_importance(final_model.feature_importance, top_n=30)
plt.savefig('../reports/feature_importance_nb.png', dpi=120)
plt.show()

# Top 10 by gain
print('Top 10 features by gain:')
display(final_model.feature_importance.head(10).reset_index().rename(columns={'index':'feature', 0:'gain'}))


## 6. Inference & Submission

In [ ]:
# --- Generate test predictions (ID-aligned to avoid ordering mismatch) ---
print('Generating test predictions…')
X_test     = test_feat[final_model.feature_names_].fillna(0)
raw_preds  = final_model.model_.predict(X_test)
test_preds = np.maximum(np.expm1(raw_preds), 0)

# Align by id — test_feat is sorted by (store_nbr, family, date) which
# differs from sample_submission.csv (date, store_nbr, family) order.
test_feat_pred = test_feat[['id']].copy()
test_feat_pred['sales'] = test_preds
submission = dfs['submission'][['id']].copy()
submission = submission.merge(test_feat_pred, on='id', how='left')
submission['sales'] = submission['sales'].clip(lower=0).fillna(0)
submission = submission[['id', 'sales']]

out_path = f'{OUT_DIR}/submission_lgbm.csv'
submission.to_csv(out_path, index=False)
print(f'Submission saved to: {out_path}')
print(f'  Rows: {len(submission):,}  Mean: {submission["sales"].mean():.2f}  Max: {submission["sales"].max():.0f}')
display(submission.head(10))


## 7. Limitations & Improvement Plan

### Current Limitations

| Issue | Impact |
|---|---|
| Lag features unavailable at test time for horizon < 16d | Requires recursive or direct multi-step strategy |
| LightGBM cannot extrapolate beyond training range | Risky for structural breaks (e.g., COVID) |
| No inter-store/cross-family correlation modelling | Misses substitution and cannibalization effects |
| Static hyperparameters (no Optuna tuning) | Sub-optimal parameters |
| No uncertainty quantification | Cannot produce prediction intervals |

### Improvement Plan

1. **Hyperparameter tuning** – Optuna + walk-forward CV objective
2. **Neural models** – N-BEATS, TFT, or TimesFM for sequence patterns
3. **Store-level micro-models** – separate LightGBM per store cluster
4. **Better multi-step strategy** – Direct multi-output regression (one model per horizon day)
5. **Ensemble** – Blend LightGBM + Prophet + Neural (weighted by CV RMSLE)
6. **Prediction intervals** – Quantile regression or conformal prediction
7. **External enrichment** – Weather data, school calendars, sports events
8. **Automated feature selection** – SHAP-based pruning of low-signal features

### Metric Discussion

**Kaggle metric: RMSLE**  
- Penalises under-prediction more than over-prediction (asymmetric)
- Works well for skewed sales distributions with many zeros
- Insensitive to large absolute errors on high-volume SKUs

**Business metrics to track additionally:**
- Weighted MAPE (by revenue share)
- Fill-rate / service level
- Inventory cost from over/under-stocking
- Bias (systematic over/under-prediction by store or family)